# MATH 5010 Computer Lab for Sections 16–17

## ANOVA and Bayesian Statistics — Full Solutions

**Course:** MATH 5010 Foundations of Statistical Theory & Probability  
**Sections:** 16–17  
**Topics:** One-way ANOVA, linear contrasts, two-way ANOVA, Bayesian inference, credible intervals, Bayesian testing, and Gibbs sampling  
**Notebook type:** Full solution version

### Learning goals

By the end of this lab, students should be able to:

1. Explain one-way ANOVA as a comparison of between-group and within-group variability.
2. Compute ANOVA sums of squares, mean squares, the $F$ statistic, and the $p$-value from scratch.
3. Interpret ANOVA in terms of a linear model.
4. Estimate and test linear contrasts among group means.
5. Simulate the sampling distribution of the ANOVA statistic under $H_0$ and under alternatives.
6. Perform basic two-way ANOVA and interpret main effects and interactions.
7. Build Bayesian posterior distributions for conjugate models.
8. Compute Bayesian point estimates, credible intervals, and posterior probabilities for tests.
9. Implement a simple Gibbs sampler for a normal model with unknown mean and variance.

> This is a full-solution notebook. For a student-facing version, remove or hide the solution cells.

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import gammaln

np.random.seed(5010)

pd.set_option("display.precision", 4)

## Section 16. Analysis of Variance (ANOVA)

### 1. The statistical problem

Suppose we observe $k$ independent groups:

$$
Y_{ij} = \mu_i + \varepsilon_{ij}, \qquad i=1,\dots,k, \quad j=1,\dots,n_i,
$$

where $\varepsilon_{ij}$ are independent normal errors with common variance $\sigma^2$.

The one-way ANOVA hypotheses are

$$
H_0: \mu_1 = \mu_2 = \cdots = \mu_k
$$

versus

$$
H_1: \text{at least one group mean is different.}
$$

The key idea is to compare:

- **between-group variation:** how far group means are from the grand mean;
- **within-group variation:** how much individual observations vary around their group means.

The ANOVA test statistic is

$$
F = \frac{MSB}{MSW}
= \frac{SSB/(k-1)}{SSW/(N-k)}.
$$

Under $H_0$ and the normal equal-variance model,

$$
F \sim F_{k-1, N-k}.
$$

## Exercise 1. One-way ANOVA from scratch

A researcher compares exam scores from three teaching methods.

- Method A: traditional lecture
- Method B: lecture plus weekly quizzes
- Method C: active learning

Use the data below.

**Tasks**

1. Compute group means and the grand mean.
2. Compute $SSB$, $SSW$, $SST$.
3. Compute $MSB$, $MSW$, and the ANOVA $F$ statistic.
4. Compute the $p$-value.
5. Compare your answer with `scipy.stats.f_oneway`.

In [ ]:
# Data
A = np.array([78, 74, 80, 76, 79, 77, 75, 81])
B = np.array([83, 85, 82, 86, 84, 87, 81, 85])
C = np.array([88, 91, 87, 90, 92, 89, 93, 86])

data = pd.DataFrame({
    "score": np.concatenate([A, B, C]),
    "method": ["A"] * len(A) + ["B"] * len(B) + ["C"] * len(C)
})

data.head()

In [ ]:
# Solution to Exercise 1

groups = [A, B, C]
group_names = ["A", "B", "C"]

N = sum(len(g) for g in groups)
k = len(groups)

group_means = np.array([g.mean() for g in groups])
group_sizes = np.array([len(g) for g in groups])
grand_mean = data["score"].mean()

SSB = sum(n_i * (mean_i - grand_mean)**2 for n_i, mean_i in zip(group_sizes, group_means))
SSW = sum(((g - g.mean())**2).sum() for g in groups)
SST = ((data["score"] - grand_mean)**2).sum()

DFB = k - 1
DFW = N - k
DFT = N - 1

MSB = SSB / DFB
MSW = SSW / DFW
F_stat = MSB / MSW
p_value = 1 - stats.f.cdf(F_stat, DFB, DFW)

anova_table = pd.DataFrame({
    "Source": ["Between groups", "Within groups", "Total"],
    "SS": [SSB, SSW, SST],
    "df": [DFB, DFW, DFT],
    "MS": [MSB, MSW, np.nan],
    "F": [F_stat, np.nan, np.nan],
    "p-value": [p_value, np.nan, np.nan]
})

print("Group means:")
for name, mean in zip(group_names, group_means):
    print(f"  Method {name}: {mean:.3f}")
print(f"Grand mean: {grand_mean:.3f}")

anova_table

In [ ]:
# Check with scipy
scipy_F, scipy_p = stats.f_oneway(A, B, C)
print(f"scipy F statistic = {scipy_F:.6f}")
print(f"scipy p-value     = {scipy_p:.6g}")

### Interpretation

The $p$-value is very small, so we reject $H_0$ at common significance levels such as $\alpha=0.05$.
There is strong evidence that not all teaching-method means are equal.

However, ANOVA does **not** directly tell us which methods differ. For that, we need contrasts or multiple comparisons.

## Exercise 2. Visualization and ANOVA intuition

Create a boxplot and overlay the individual observations. Then explain why the ANOVA statistic is large or small for this dataset.

In [ ]:
# Solution to Exercise 2

fig, ax = plt.subplots(figsize=(7, 5))
positions = np.arange(1, k + 1)
ax.boxplot(groups, positions=positions, widths=0.55)

for pos, g in zip(positions, groups):
    jitter = np.random.normal(0, 0.04, size=len(g))
    ax.scatter(np.full(len(g), pos) + jitter, g, alpha=0.8)

ax.set_xticks(positions)
ax.set_xticklabels(["Method A", "Method B", "Method C"])
ax.set_ylabel("Score")
ax.set_title("Exam scores by teaching method")
ax.grid(axis="y", alpha=0.3)
plt.show()

print("Solution explanation:")
print("The group means are separated relative to the within-group spread.")
print("Therefore SSB is large compared with SSW, so MSB/MSW is large.")

## Exercise 3. Simulating the null distribution of the ANOVA statistic

Assume the null hypothesis is true, so all groups have the same mean and variance.

**Tasks**

1. Simulate three groups of size 8 from the same normal distribution.
2. Compute the ANOVA $F$ statistic.
3. Repeat many times.
4. Compare the simulated distribution with the theoretical $F_{2,21}$ distribution.

In [ ]:
# Solution to Exercise 3

def one_way_anova_F(*groups):
    k = len(groups)
    N = sum(len(g) for g in groups)
    all_values = np.concatenate(groups)
    grand_mean = all_values.mean()
    SSB = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
    SSW = sum(((g - g.mean())**2).sum() for g in groups)
    MSB = SSB / (k - 1)
    MSW = SSW / (N - k)
    return MSB / MSW

R = 10000
F_null = np.empty(R)

for r in range(R):
    g1 = np.random.normal(loc=80, scale=5, size=8)
    g2 = np.random.normal(loc=80, scale=5, size=8)
    g3 = np.random.normal(loc=80, scale=5, size=8)
    F_null[r] = one_way_anova_F(g1, g2, g3)

x = np.linspace(0, 8, 400)

fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(F_null, bins=50, density=True, alpha=0.6, label="Simulated under $H_0$")
ax.plot(x, stats.f.pdf(x, 2, 21), linewidth=2, label="$F_{2,21}$ density")
ax.axvline(F_stat, linestyle="--", linewidth=2, label="Observed $F$")
ax.set_xlabel("F statistic")
ax.set_ylabel("Density")
ax.set_title("Null distribution of one-way ANOVA F statistic")
ax.legend()
plt.show()

print(f"Empirical P(F >= observed F): {np.mean(F_null >= F_stat):.6f}")
print(f"Theoretical p-value:          {p_value:.6f}")

## Exercise 4. Power of ANOVA by simulation

Power is the probability of rejecting $H_0$ when $H_1$ is true.

Suppose the three true means are

$$
\mu_A = 80, \qquad \mu_B = 80+d, \qquad \mu_C = 80+2d,
$$

with common standard deviation $\sigma=5$ and sample size $n=8$ per group.

Estimate the power of ANOVA at significance level $\alpha=0.05$ for several values of $d$.

In [ ]:
# Solution to Exercise 4

def simulate_anova_power(d, n=8, sigma=5, alpha=0.05, R=5000):
    rejections = 0
    for _ in range(R):
        g1 = np.random.normal(80, sigma, n)
        g2 = np.random.normal(80 + d, sigma, n)
        g3 = np.random.normal(80 + 2*d, sigma, n)
        F = one_way_anova_F(g1, g2, g3)
        p = 1 - stats.f.cdf(F, 2, 3*n - 3)
        rejections += (p < alpha)
    return rejections / R

d_values = np.arange(0, 7.5, 0.5)
powers = np.array([simulate_anova_power(d) for d in d_values])

power_table = pd.DataFrame({"difference_step_d": d_values, "estimated_power": powers})
display(power_table.head(10))

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(d_values, powers, marker="o")
ax.axhline(0.05, linestyle="--", label="alpha = 0.05")
ax.set_xlabel("Effect step $d$")
ax.set_ylabel("Estimated power")
ax.set_ylim(0, 1.05)
ax.set_title("ANOVA power increases as group means separate")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## Exercise 5. Linear contrasts

ANOVA tells us whether there is evidence that at least one mean differs. A **linear contrast** asks a more specific question.

A contrast has the form

$$
L = c_1 \mu_1 + c_2 \mu_2 + \cdots + c_k \mu_k,
$$

where

$$
\sum_{i=1}^k c_i = 0.
$$

For example, to compare Method C with the average of Methods A and B, use

$$
L = \mu_C - \frac{\mu_A + \mu_B}{2}.
$$

The coefficients are

$$
(c_A,c_B,c_C)=\left(-\frac12,-\frac12,1\right).
$$

**Tasks**

1. Estimate this contrast.
2. Compute its standard error using $MSW$.
3. Compute a $t$ statistic and two-sided $p$-value.
4. Compute a 95% confidence interval.

In [ ]:
# Solution to Exercise 5

c = np.array([-0.5, -0.5, 1.0])

contrast_est = np.sum(c * group_means)
contrast_se = np.sqrt(MSW * np.sum(c**2 / group_sizes))
t_stat = contrast_est / contrast_se
contrast_df = DFW
contrast_p = 2 * (1 - stats.t.cdf(abs(t_stat), df=contrast_df))
crit = stats.t.ppf(0.975, df=contrast_df)
ci_low = contrast_est - crit * contrast_se
ci_high = contrast_est + crit * contrast_se

contrast_table = pd.DataFrame({
    "contrast": ["C - average(A,B)"],
    "estimate": [contrast_est],
    "SE": [contrast_se],
    "t": [t_stat],
    "df": [contrast_df],
    "p-value": [contrast_p],
    "CI lower": [ci_low],
    "CI upper": [ci_high]
})

contrast_table

### Interpretation

The contrast is positive because Method C has a larger sample mean than the average of Methods A and B.
The confidence interval is above 0, so there is strong evidence that Method C outperforms the average of Methods A and B in this dataset.

## Exercise 6. Two-way ANOVA with interaction

Now suppose exam scores depend on two factors:

- teaching method: A, B, C;
- study format: individual or group.

A two-way ANOVA model with interaction is

$$
Y_{ijk} = \mu + \alpha_i + \beta_j + (\alpha\beta)_{ij} + \varepsilon_{ijk}.
$$

**Tasks**

1. Simulate a balanced two-way dataset.
2. Fit the model using least squares.
3. Compute sums of squares for method, format, interaction, and error.
4. Interpret the results.

We will compute a balanced two-way ANOVA from scratch using the decomposition:

$$
SST = SS_A + SS_B + SS_{AB} + SSE.
$$

In [ ]:
# Solution to Exercise 6

np.random.seed(16)
methods = ["A", "B", "C"]
formats = ["Individual", "Group"]
rep = 6

# True means: rows are methods, columns are formats.
true_means = {
    ("A", "Individual"): 78,
    ("A", "Group"): 80,
    ("B", "Individual"): 83,
    ("B", "Group"): 85,
    ("C", "Individual"): 87,
    ("C", "Group"): 93,  # larger group-learning gain for method C creates interaction
}

rows = []
for m in methods:
    for f in formats:
        values = np.random.normal(true_means[(m, f)], 3, rep)
        for y in values:
            rows.append({"method": m, "format": f, "score": y})

df2 = pd.DataFrame(rows)
display(df2.head())

cell_means = df2.groupby(["method", "format"])["score"].mean().unstack()
display(cell_means)

In [ ]:
# Balanced two-way ANOVA from scratch

a = len(methods)
b = len(formats)
n = rep
N2 = len(df2)

grand = df2["score"].mean()
method_means = df2.groupby("method")["score"].mean()
format_means = df2.groupby("format")["score"].mean()
cell_means_series = df2.groupby(["method", "format"])["score"].mean()

SST2 = ((df2["score"] - grand)**2).sum()
SSA = b * n * sum((method_means[m] - grand)**2 for m in methods)
SSB2 = a * n * sum((format_means[f] - grand)**2 for f in formats)
SSAB = 0.0
for m in methods:
    for f in formats:
        interaction_part = cell_means_series[(m, f)] - method_means[m] - format_means[f] + grand
        SSAB += n * interaction_part**2
SSE = 0.0
for m in methods:
    for f in formats:
        vals = df2[(df2["method"] == m) & (df2["format"] == f)]["score"].values
        cm = cell_means_series[(m, f)]
        SSE += ((vals - cm)**2).sum()

df_A = a - 1
df_B = b - 1
df_AB = (a - 1) * (b - 1)
df_E = a * b * (n - 1)
df_T = N2 - 1

MSA = SSA / df_A
MSB2 = SSB2 / df_B
MSAB = SSAB / df_AB
MSE = SSE / df_E

F_A = MSA / MSE
F_B = MSB2 / MSE
F_AB = MSAB / MSE

p_A = 1 - stats.f.cdf(F_A, df_A, df_E)
p_B = 1 - stats.f.cdf(F_B, df_B, df_E)
p_AB = 1 - stats.f.cdf(F_AB, df_AB, df_E)

two_way_table = pd.DataFrame({
    "Source": ["Method", "Format", "Method x Format", "Error", "Total"],
    "SS": [SSA, SSB2, SSAB, SSE, SST2],
    "df": [df_A, df_B, df_AB, df_E, df_T],
    "MS": [MSA, MSB2, MSAB, MSE, np.nan],
    "F": [F_A, F_B, F_AB, np.nan, np.nan],
    "p-value": [p_A, p_B, p_AB, np.nan, np.nan]
})

two_way_table

In [ ]:
# Interaction plot

fig, ax = plt.subplots(figsize=(7, 5))
for m in methods:
    y = [cell_means.loc[m, f] for f in formats]
    ax.plot(formats, y, marker="o", label=f"Method {m}")
ax.set_ylabel("Mean score")
ax.set_title("Interaction plot: method by study format")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

print("Solution interpretation:")
print("A significant method effect means average scores differ across teaching methods.")
print("A significant format effect means individual vs group study differs on average.")
print("A significant interaction means the effect of format depends on method.")

## Section 17. Bayesian Statistics

### 2. Bayesian inference framework

Bayesian inference begins with a prior distribution $\pi(\theta)$ and updates it using the likelihood $L(\theta)$:

$$
\pi(\theta \mid x) \propto L(\theta; x)\pi(\theta).
$$

Bayesian outputs include:

- posterior distribution;
- posterior mean or median as point estimates;
- credible intervals;
- posterior probabilities of hypotheses;
- posterior predictive distributions.

A Bayesian credible interval is different from a frequentist confidence interval. A 95% credible interval means that, after observing the data and using the prior, the posterior probability that $\theta$ lies in the interval is 0.95.

## Exercise 7. Beta-Binomial model for a conversion rate

Suppose an A/B test observes $x=42$ conversions out of $n=100$ visitors.
Let the conversion probability be $p$.

Assume the prior

$$
p \sim \text{Beta}(a,b).
$$

The likelihood is

$$
X \mid p \sim \text{Binomial}(n,p).
$$

The posterior is

$$
p \mid X=x \sim \text{Beta}(a+x,b+n-x).
$$

Use prior $\text{Beta}(2,2)$.

**Tasks**

1. Find the posterior distribution.
2. Compute the posterior mean.
3. Compute a 95% credible interval.
4. Compute $P(p>0.40 \mid x)$.
5. Compare with the MLE $\hat p=x/n$.

In [ ]:
# Solution to Exercise 7

x_obs = 42
n_obs = 100
a_prior, b_prior = 2, 2

a_post = a_prior + x_obs
b_post = b_prior + n_obs - x_obs

posterior_mean = a_post / (a_post + b_post)
posterior_median = stats.beta.ppf(0.5, a_post, b_post)
cred_int = stats.beta.ppf([0.025, 0.975], a_post, b_post)
prob_gt_040 = 1 - stats.beta.cdf(0.40, a_post, b_post)
mle_p = x_obs / n_obs

beta_binomial_summary = pd.DataFrame({
    "quantity": ["posterior alpha", "posterior beta", "MLE", "posterior mean", "posterior median", "CI lower", "CI upper", "P(p > 0.40 | data)"],
    "value": [a_post, b_post, mle_p, posterior_mean, posterior_median, cred_int[0], cred_int[1], prob_gt_040]
})

beta_binomial_summary

In [ ]:
# Plot prior and posterior

grid = np.linspace(0, 1, 500)
prior_density = stats.beta.pdf(grid, a_prior, b_prior)
post_density = stats.beta.pdf(grid, a_post, b_post)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(grid, prior_density, label="Prior Beta(2,2)")
ax.plot(grid, post_density, label=f"Posterior Beta({a_post},{b_post})")
ax.axvline(mle_p, linestyle="--", label="MLE")
ax.axvline(posterior_mean, linestyle=":", label="Posterior mean")
ax.set_xlabel("p")
ax.set_ylabel("Density")
ax.set_title("Beta-Binomial Bayesian update")
ax.legend()
plt.show()

### Solution interpretation

The posterior is

$$
p \mid X=42 \sim \text{Beta}(44,60).
$$

The posterior mean is slightly shrunk toward the prior mean $0.5$ compared with the MLE $0.42$.
The posterior probability $P(p>0.40 \mid x)$ directly answers a probability question about the unknown parameter.

## Exercise 8. Gamma-Poisson model for a count rate

Suppose daily customer arrivals follow

$$
X_i \mid \lambda \sim \text{Poisson}(\lambda), \qquad i=1,\dots,n.
$$

Use the Gamma prior with shape-rate parameterization:

$$
\lambda \sim \text{Gamma}(a,b),
$$

where $E[\lambda]=a/b$.

The posterior is

$$
\lambda \mid x_1,\dots,x_n \sim \text{Gamma}\left(a+\sum_{i=1}^n x_i,\ b+n\right).
$$

**Tasks**

1. Simulate 20 daily counts with true $\lambda=6$.
2. Use prior $\lambda\sim\text{Gamma}(2,1)$.
3. Compute the posterior distribution.
4. Compute the posterior mean and 95% credible interval.
5. Compute $P(\lambda>5 \mid \text{data})$.
6. Compare the Bayesian estimate with the MLE $\bar X$.

In [ ]:
# Solution to Exercise 8

np.random.seed(17)
true_lambda = 6
counts = np.random.poisson(true_lambda, size=20)
S = counts.sum()
n = len(counts)

a0, b0 = 2, 1  # shape, rate
aN = a0 + S
bN = b0 + n

lambda_mle = counts.mean()
lambda_post_mean = aN / bN
lambda_post_ci = stats.gamma.ppf([0.025, 0.975], a=aN, scale=1/bN)
prob_lambda_gt_5 = 1 - stats.gamma.cdf(5, a=aN, scale=1/bN)

poisson_bayes_summary = pd.DataFrame({
    "quantity": ["n", "sum counts", "MLE", "posterior shape", "posterior rate", "posterior mean", "CI lower", "CI upper", "P(lambda > 5 | data)"],
    "value": [n, S, lambda_mle, aN, bN, lambda_post_mean, lambda_post_ci[0], lambda_post_ci[1], prob_lambda_gt_5]
})

print("Counts:", counts)
poisson_bayes_summary

In [ ]:
# Plot prior and posterior for lambda

grid = np.linspace(0, 12, 500)
prior = stats.gamma.pdf(grid, a=a0, scale=1/b0)
posterior = stats.gamma.pdf(grid, a=aN, scale=1/bN)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(grid, prior, label=f"Prior Gamma({a0},{b0})")
ax.plot(grid, posterior, label=f"Posterior Gamma({aN},{bN})")
ax.axvline(lambda_mle, linestyle="--", label="MLE")
ax.axvline(lambda_post_mean, linestyle=":", label="Posterior mean")
ax.set_xlabel("lambda")
ax.set_ylabel("Density")
ax.set_title("Gamma-Poisson Bayesian update")
ax.legend()
plt.show()

## Exercise 9. Bayesian one-sided test for a Poisson mean

Suppose we observe a Poisson sample with total count $S=130$ over $n=20$ observations.
Use prior

$$
\lambda \sim \text{Gamma}(a=2,b=1).
$$

Test

$$
H_0: \lambda \le 5 \qquad \text{versus} \qquad H_1: \lambda > 5.
$$

Use the Bayesian decision rule:

$$
\text{Reject }H_0 \quad \text{if} \quad P(H_0 \mid X) < 0.05.
$$

**Task:** Compute $P(\lambda\le 5\mid X)$ and make the decision.

In [ ]:
# Solution to Exercise 9

n_test = 20
S_test = 130
a_test, b_test = 2, 1
lambda0 = 5

a_post_test = a_test + S_test
b_post_test = b_test + n_test

post_prob_H0 = stats.gamma.cdf(lambda0, a=a_post_test, scale=1/b_post_test)
post_prob_H1 = 1 - post_prob_H0
post_mean_test = a_post_test / b_post_test
post_ci_test = stats.gamma.ppf([0.025, 0.975], a=a_post_test, scale=1/b_post_test)

decision = "Reject H0" if post_prob_H0 < 0.05 else "Fail to reject H0"

pd.DataFrame({
    "quantity": ["posterior", "posterior mean", "95% CI lower", "95% CI upper", "P(H0 | data)", "P(H1 | data)", "decision"],
    "value": [f"Gamma({a_post_test}, {b_post_test})", post_mean_test, post_ci_test[0], post_ci_test[1], post_prob_H0, post_prob_H1, decision]
})

### Interpretation

The posterior mean is greater than 5, and the posterior probability that $\lambda\le 5$ is very small.
Under the rule $P(H_0\mid X)<0.05$, we reject $H_0$.

## Exercise 10. Bayesian normal mean with known variance

Suppose

$$
X_i \mid \mu \sim N(\mu,\sigma^2), \qquad \sigma^2 \text{ known},
$$

and the prior is

$$
\mu \sim N(m_0,s_0^2).
$$

The posterior is normal:

$$
\mu \mid x \sim N(m_n,s_n^2),
$$

where

$$
s_n^2 = \left(\frac{1}{s_0^2}+\frac{n}{\sigma^2}\right)^{-1}
$$

and

$$
m_n = s_n^2\left(\frac{m_0}{s_0^2}+\frac{n\bar x}{\sigma^2}\right).
$$

**Tasks**

1. Simulate $n=30$ observations from $N(75,10^2)$.
2. Use prior $\mu\sim N(70,15^2)$.
3. Compute the posterior mean and standard deviation.
4. Compute a 95% credible interval.
5. Compute $P(\mu>72\mid x)$.

In [ ]:
# Solution to Exercise 10

np.random.seed(100)
mu_true = 75
sigma_known = 10
x_normal = np.random.normal(mu_true, sigma_known, size=30)

m0 = 70
s0 = 15
n = len(x_normal)
xbar = x_normal.mean()

sn2 = 1 / (1/s0**2 + n/sigma_known**2)
sn = np.sqrt(sn2)
mn = sn2 * (m0/s0**2 + n*xbar/sigma_known**2)

mu_ci = stats.norm.ppf([0.025, 0.975], loc=mn, scale=sn)
prob_mu_gt_72 = 1 - stats.norm.cdf(72, loc=mn, scale=sn)

normal_known_summary = pd.DataFrame({
    "quantity": ["sample mean", "posterior mean", "posterior sd", "CI lower", "CI upper", "P(mu > 72 | data)"],
    "value": [xbar, mn, sn, mu_ci[0], mu_ci[1], prob_mu_gt_72]
})

normal_known_summary

In [ ]:
# Plot likelihood shape, prior, posterior on a comparable scale

grid = np.linspace(60, 90, 500)
prior_mu = stats.norm.pdf(grid, m0, s0)
post_mu = stats.norm.pdf(grid, mn, sn)
# sampling distribution of xbar as function of mu, normalized as density in mu
like_mu = stats.norm.pdf(xbar, loc=grid, scale=sigma_known/np.sqrt(n))
like_mu = like_mu / np.trapz(like_mu, grid)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(grid, prior_mu, label="Prior for mu")
ax.plot(grid, like_mu, label="Normalized likelihood shape")
ax.plot(grid, post_mu, label="Posterior for mu")
ax.axvline(xbar, linestyle="--", label="Sample mean")
ax.set_xlabel("mu")
ax.set_ylabel("Density")
ax.set_title("Normal-normal Bayesian update")
ax.legend()
plt.show()

## Exercise 11. Gibbs sampler for normal mean and variance

Now consider a Bayesian normal model with both $\mu$ and $\sigma^2$ unknown:

$$
X_i \mid \mu,\sigma^2 \sim N(\mu,\sigma^2).
$$

Use the conjugate prior:

$$
\mu \mid \sigma^2 \sim N\left(m_0,\frac{\sigma^2}{\kappa_0}\right),
$$

and

$$
\sigma^2 \sim \text{Inv-Gamma}(\alpha_0,\beta_0).
$$

The full conditional distributions are:

$$
\mu \mid \sigma^2,x \sim N\left(m_n,\frac{\sigma^2}{\kappa_n}\right),
$$

where

$$
\kappa_n = \kappa_0+n,
\qquad
m_n = \frac{\kappa_0m_0+n\bar x}{\kappa_0+n},
$$

and

$$
\sigma^2 \mid \mu,x \sim \text{Inv-Gamma}\left(\alpha_0+\frac n2,\ \beta_0+\frac12\sum_{i=1}^n(x_i-\mu)^2+\frac{\kappa_0}{2}(\mu-m_0)^2\right).
$$

**Tasks**

1. Simulate normal data.
2. Implement the Gibbs sampler.
3. Plot trace plots for $\mu$ and $\sigma^2$.
4. Estimate posterior means and 95% credible intervals.

In [ ]:
# Solution to Exercise 11

np.random.seed(123)
x_gibbs = np.random.normal(loc=10, scale=2, size=40)
n = len(x_gibbs)
xbar = x_gibbs.mean()

# Prior hyperparameters
m0 = 0.0
kappa0 = 0.01
alpha0 = 2.0
beta0 = 2.0

# Gibbs settings
n_iter = 12000
burn = 2000

mu_samples = np.empty(n_iter)
sigma2_samples = np.empty(n_iter)

# Initialize
mu_current = xbar
sigma2_current = np.var(x_gibbs, ddof=1)

kappa_n = kappa0 + n
m_n = (kappa0 * m0 + n * xbar) / kappa_n
alpha_n = alpha0 + n / 2

for t in range(n_iter):
    # Sample mu | sigma^2, x
    mu_sd = np.sqrt(sigma2_current / kappa_n)
    mu_current = np.random.normal(m_n, mu_sd)
    
    # Sample sigma^2 | mu, x
    beta_n_mu = beta0 + 0.5 * np.sum((x_gibbs - mu_current)**2) + 0.5 * kappa0 * (mu_current - m0)**2
    # If V ~ Inv-Gamma(alpha, beta), then 1/V ~ Gamma(alpha, rate=beta)
    precision = np.random.gamma(shape=alpha_n, scale=1/beta_n_mu)
    sigma2_current = 1 / precision
    
    mu_samples[t] = mu_current
    sigma2_samples[t] = sigma2_current

mu_post = mu_samples[burn:]
sigma2_post = sigma2_samples[burn:]

summary_gibbs = pd.DataFrame({
    "parameter": ["mu", "sigma^2", "sigma"],
    "posterior mean": [mu_post.mean(), sigma2_post.mean(), np.sqrt(sigma2_post).mean()],
    "2.5%": [np.quantile(mu_post, 0.025), np.quantile(sigma2_post, 0.025), np.quantile(np.sqrt(sigma2_post), 0.025)],
    "97.5%": [np.quantile(mu_post, 0.975), np.quantile(sigma2_post, 0.975), np.quantile(np.sqrt(sigma2_post), 0.975)]
})

print(f"Sample mean = {xbar:.3f}")
print(f"Sample sd   = {np.std(x_gibbs, ddof=1):.3f}")
summary_gibbs

In [ ]:
# Trace plot for mu

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(mu_samples, linewidth=0.7)
ax.axvline(burn, linestyle="--", label="burn-in cutoff")
ax.set_xlabel("Iteration")
ax.set_ylabel("mu")
ax.set_title("Gibbs trace plot for mu")
ax.legend()
plt.show()

In [ ]:
# Trace plot for sigma^2

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sigma2_samples, linewidth=0.7)
ax.axvline(burn, linestyle="--", label="burn-in cutoff")
ax.set_xlabel("Iteration")
ax.set_ylabel("sigma^2")
ax.set_title("Gibbs trace plot for sigma^2")
ax.legend()
plt.show()

In [ ]:
# Posterior histograms

fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(mu_post, bins=40, density=True, alpha=0.7)
ax.axvline(mu_post.mean(), linestyle="--", label="posterior mean")
ax.set_xlabel("mu")
ax.set_ylabel("Density")
ax.set_title("Posterior samples for mu")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(sigma2_post, bins=40, density=True, alpha=0.7)
ax.axvline(sigma2_post.mean(), linestyle="--", label="posterior mean")
ax.set_xlabel("sigma^2")
ax.set_ylabel("Density")
ax.set_title("Posterior samples for sigma^2")
ax.legend()
plt.show()

### Solution interpretation

The Gibbs sampler alternates between two easy conditional distributions:

1. sample $\mu$ given $\sigma^2$ and the data;
2. sample $\sigma^2$ given $\mu$ and the data.

After burn-in, the draws behave like dependent samples from the joint posterior distribution.
The posterior mean of $\mu$ should be close to the sample mean, and the posterior distribution of $\sigma$ should be close to the sample standard deviation, with uncertainty due to finite sample size.

## Exercise 12. Posterior predictive distribution

A Bayesian analysis can predict a future observation by averaging over posterior uncertainty.

For the Gibbs samples from Exercise 11, a posterior predictive draw can be generated by:

1. draw $(\mu^{(s)},\sigma^{2(s)})$ from the posterior samples;
2. draw

$$
X_{new}^{(s)} \sim N(\mu^{(s)},\sigma^{2(s)}).
$$

**Tasks**

1. Generate posterior predictive samples.
2. Compute a 95% posterior predictive interval.
3. Compare it with the posterior credible interval for $\mu$.

In [ ]:
# Solution to Exercise 12

S_post = len(mu_post)
pred_samples = np.random.normal(mu_post, np.sqrt(sigma2_post))

pred_interval = np.quantile(pred_samples, [0.025, 0.975])
mu_interval = np.quantile(mu_post, [0.025, 0.975])

predictive_summary = pd.DataFrame({
    "interval": ["credible interval for mu", "predictive interval for new X"],
    "lower": [mu_interval[0], pred_interval[0]],
    "upper": [mu_interval[1], pred_interval[1]],
    "width": [mu_interval[1] - mu_interval[0], pred_interval[1] - pred_interval[0]]
})

predictive_summary

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(pred_samples, bins=40, density=True, alpha=0.6, label="posterior predictive samples")
ax.hist(mu_post, bins=40, density=True, alpha=0.5, label="posterior samples of mu")
ax.set_xlabel("Value")
ax.set_ylabel("Density")
ax.set_title("Posterior for mean vs posterior predictive distribution")
ax.legend()
plt.show()

print("Solution explanation:")
print("The predictive interval is wider because it includes both parameter uncertainty and future observation noise.")
print("The credible interval for mu includes only uncertainty about the unknown mean parameter.")

## Lab summary

In this lab, we practiced two major topics from the final part of MATH 5010.

### ANOVA

The one-way ANOVA model compares several means using the statistic

$$
F=\frac{SSB/(k-1)}{SSW/(N-k)}.
$$

A large $F$ means that group means are far apart relative to within-group variation.
We also studied simulation, power, contrasts, and two-way ANOVA with interaction.

### Bayesian inference

Bayesian inference updates prior beliefs using the likelihood:

$$
\pi(\theta\mid x)\propto L(\theta;x)\pi(\theta).
$$

We studied Beta-Binomial, Gamma-Poisson, normal-normal, Bayesian hypothesis testing, Gibbs sampling, and posterior predictive distributions.

### Key conceptual comparison

| Topic | Frequentist interpretation | Bayesian interpretation |
|---|---|---|
| Parameter | Fixed unknown constant | Unknown quantity described by a posterior distribution |
| Interval | Confidence interval from repeated sampling | Credible interval with posterior probability |
| Test | Reject/fail to reject using $p$-value | Compute posterior probability of hypotheses |
| Prediction | Often plug in estimated parameters | Average over posterior uncertainty |

## Additional challenge problems with solutions

### Challenge 1. ANOVA identity

Show numerically that

$$
SST = SSB + SSW.
$$

### Challenge 2. Posterior sensitivity

Repeat the Beta-Binomial analysis with priors $\text{Beta}(1,1)$, $\text{Beta}(10,10)$, and $\text{Beta}(40,60)$. Compare the posterior means.

In [ ]:
# Challenge 1 solution

print(f"SST       = {SST:.10f}")
print(f"SSB + SSW = {(SSB + SSW):.10f}")
print(f"Difference = {SST - SSB - SSW:.10e}")

In [ ]:
# Challenge 2 solution

priors = [(1, 1), (10, 10), (40, 60)]
rows = []
for aa, bb in priors:
    ap = aa + x_obs
    bp = bb + n_obs - x_obs
    rows.append({
        "prior": f"Beta({aa},{bb})",
        "prior mean": aa/(aa+bb),
        "posterior": f"Beta({ap},{bp})",
        "posterior mean": ap/(ap+bp),
        "95% lower": stats.beta.ppf(0.025, ap, bp),
        "95% upper": stats.beta.ppf(0.975, ap, bp)
    })

sensitivity = pd.DataFrame(rows)
sensitivity

### Sensitivity interpretation

When the prior is weak, the posterior mean is close to the sample proportion.
When the prior is stronger, the posterior mean is pulled more strongly toward the prior mean.
This is not a bug; it is the Bayesian mechanism for combining prior information with data.